#Import

In [0]:
# Import required PySpark libraries for data transformation
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import date

#Get Data From Bronze

In [0]:
# Read customer data from the Bronze layer in ADLS Gen2
df = spark.read.format("parquet").option("inferSchema",True)\
        .load("abfss://bronze@storenapatdatalake.dfs.core.windows.net/crm_cust_info")

In [0]:
display(df.limit(5))

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,M,M,2025-10-06
11001,AW00011001,Eugene,Huang,S,M,2025-10-06
11002,AW00011002,Ruben,Torres,M,M,2025-10-06
11003,AW00011003,Christy,Zhu,S,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06


#Data Transformation

##Trimming 

In [0]:
# Trim whitespace from customer attributes
df = df.withColumn("cst_key",trim(col("cst_key")))\
    .withColumn("cst_firstname",trim(col("cst_firstname")))\
    .withColumn("cst_lastname",trim(col("cst_lastname")))\
    .withColumn("cst_marital_status",trim(col("cst_marital_status")))\
    .withColumn("cst_gndr",trim(col("cst_gndr")))
df.limit(10).display()

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,M,M,2025-10-06
11001,AW00011001,Eugene,Huang,S,M,2025-10-06
11002,AW00011002,Ruben,Torres,M,M,2025-10-06
11003,AW00011003,Christy,Zhu,S,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06
11005,AW00011005,Julio,Ruiz,S,M,2025-10-06
11006,AW00011006,Janet,Alvarez,S,F,2025-10-06
11007,AW00011007,Marco,Mehta,M,M,2025-10-06
11008,AW00011008,Rob,Verhoff,S,F,2025-10-06
11009,AW00011009,Shannon,Carlson,S,M,2025-10-06


##Product Line Normalization

In [0]:
# Standardize marital status codes into readable values
df = df.withColumn("cst_marital_status",
            when(col("cst_marital_status") == "M","Married")\
            .when(col("cst_marital_status") == "S","Single")\
            .otherwise("n/a"))
df.limit(10).display()

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,Married,M,2025-10-06
11001,AW00011001,Eugene,Huang,Single,M,2025-10-06
11002,AW00011002,Ruben,Torres,Married,M,2025-10-06
11003,AW00011003,Christy,Zhu,Single,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,Single,F,2025-10-06
11005,AW00011005,Julio,Ruiz,Single,M,2025-10-06
11006,AW00011006,Janet,Alvarez,Single,F,2025-10-06
11007,AW00011007,Marco,Mehta,Married,M,2025-10-06
11008,AW00011008,Rob,Verhoff,Single,F,2025-10-06
11009,AW00011009,Shannon,Carlson,Single,M,2025-10-06


In [0]:
# Standardize gender codes into readable values
df = df.withColumn("cst_gndr",
            when(col("cst_gndr") == "M","Male")\
            .when(col("cst_gndr") == "F","Female")\
            .otherwise("n/a"))
df.limit(10).display()

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06
11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06
11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06
11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
11005,AW00011005,Julio,Ruiz,Single,Male,2025-10-06
11006,AW00011006,Janet,Alvarez,Single,Female,2025-10-06
11007,AW00011007,Marco,Mehta,Married,Male,2025-10-06
11008,AW00011008,Rob,Verhoff,Single,Female,2025-10-06
11009,AW00011009,Shannon,Carlson,Single,Male,2025-10-06


##Filter

In [0]:
# Filter records with valid customer key length
df = df.filter(length(col("cst_key")) == 10)

#Rename Columns

In [0]:
# Rename columns to business-friendly names
RENAME_MAP = {
    "cst_id" : "customer_id",
    "cst_key" : "customer_key",
    "cst_firstname" : "first_name",
    "cst_lastname" : "last_name",
    "cst_marital_status" : "marital_status",
    "cst_gndr" : "gender",
    "cst_create_date" : "create_date"
}

for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

##Create silver record_priority to filter out duplicate id

In [0]:
# Rank customer records by ID and latest creation date
w = Window.partitionBy(df["customer_id"]).orderBy(df["create_date"].desc_nulls_last())
df = df.withColumn("record_priority",row_number().over(w))

In [0]:
df.limit(5).display()

customer_id,customer_key,first_name,last_name,marital_status,gender,create_date,record_priority
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06,1
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06,1
11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06,1
11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06,1
11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06,1


In [0]:
# Remove duplicate customer records, keeping the highest priority
df = df.filter(col("record_priority") == 1)

#Test

#Data Writing

In [0]:
# Write cleaned customer data to the Silver layer in Parquet format
df.write.format("parquet")\
    .mode("overwrite")\
    .option("path","abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_cust_info")\
    .save()

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_cust_info` LIMIT 10

customer_id,customer_key,first_name,last_name,marital_status,gender,create_date,record_priority
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06,1
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06,1
11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06,1
11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06,1
11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06,1
11005,AW00011005,Julio,Ruiz,Single,Male,2025-10-06,1
11006,AW00011006,Janet,Alvarez,Single,Female,2025-10-06,1
11007,AW00011007,Marco,Mehta,Married,Male,2025-10-06,1
11008,AW00011008,Rob,Verhoff,Single,Female,2025-10-06,1
11009,AW00011009,Shannon,Carlson,Single,Male,2025-10-06,1


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_cust_info` WHERE record_priority > 1 ORDER BY record_priority DESC LIMIT 10

customer_id,customer_key,first_name,last_name,marital_status,gender,create_date,record_priority
